# Exp3 — размер batch после общего SB-предобучения

Чистая версия для проекта `AI360_group23_autumn_2026`. Помести notebook в
`experiments/exp3/` распакованного проекта. Запускай ячейки сверху вниз.
Датасет: CIFAR-10, 45 000 train / 5 000 validation; официальный test не используется для подбора.

Один warmup-checkpoint → независимые продолжения **constant**, **exponential** (или linear), **immediate**.
По умолчанию: 10 + 100 эпох, SB 256, LB 10% train, ADAM, постоянный LR 3e-4.
Сохраняются train/validation, время, число обновлений, checkpoint для возобновления и графики.

Модель импортируется из `models.instances.ShallowNN`. Профиль `notebook_c1` сохраняет
SAME pooling, Xavier-инициализацию и batch statistics из последней C1 исходного notebook.
Профиль `project_native` использует неизменённую модель архива. Это разные протоколы;
не смешивай их результаты. Перенос не является точным воспроизведением старого Keras mode=2.

Длительное обучение запускается только в отдельной ячейке «Запуск / продолжение».
Дополнительные проверки BN, памяти и sharpness выключены по умолчанию.
Старые outputs очищены: они могли относиться к другим определениям функций.


### 1. Зависимости (при необходимости)

Если пакеты уже установлены, ничего выполнять не нужно. Для CUDA выбери подходящую сборку PyTorch. В requirements проекта отсутствуют torchvision и pandas, необходимые notebook.


In [ ]:
# %pip install torch torchvision numpy pandas matplotlib


### 2. Импорты и корень проекта

Ищем реальный корень проекта по models/ и eval.py. Если notebook лежит вне проекта, укажи PROJECT_ROOT_OVERRIDE. Файлы проекта не изменяются.


In [ ]:
import copy
import gc
import hashlib
import importlib
import json
import random
import sys
import time
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT_OVERRIDE = None  # например Path('/home/jupyter/project/AI360_group23_autumn_2026')
candidates = ([Path(PROJECT_ROOT_OVERRIDE).expanduser()] if PROJECT_ROOT_OVERRIDE
              else [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
    if (p / 'models/instances/ShallowNN.py').is_file() and (p / 'eval.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Распакуй проект и укажи PROJECT_ROOT_OVERRIDE.')
sys.path.insert(0, str(PROJECT_ROOT))
from models.instances.ShallowNN import ShallowNN
from models.instances.CNN import CNN
from models.instances.DNN import DNN
from eval import evaluate as project_evaluate
from metrics import Accuracy

for module_name in ('models.instances.ShallowNN', 'eval', 'metrics'):
    location = Path(importlib.import_module(module_name).__file__).resolve()
    if not location.is_relative_to(PROJECT_ROOT):
        raise ImportError(f'{module_name} загружен не из проекта: {location}. Перезапусти kernel.')
print('Проект:', PROJECT_ROOT)


### 3. Единая конфигурация

Все параметры меняются здесь. smoke=True — быстрый технический прогон на случайных данных. Для продолжения после сбоя укажи существующую папку в RESUME_DIR; не меняй параметры эксперимента. По умолчанию новое обучение разрешено только явным вызовом ниже.


In [ ]:
CFG = {
    'model': 'ShallowNN',  # ShallowNN, CNN, DNN, legacy_vgg11_bn
    'model_profile': 'notebook_c1',  # notebook_c1 или project_native; профиль только для ShallowNN
    'smoke': False,
    'data_root': str(PROJECT_ROOT / 'datasets'),
    'download': False,
    'val_size': 5000,
    'split_seed': 123,
    'num_classes': 10,
    'warmup_epochs': 10,
    'continue_epochs': 100,
    'batch_start': 256,
    'batch_end': None,  # None → lb_fraction от train; либо конкретное число
    'lb_fraction': 0.10,
    'branches': ['constant', 'exponential', 'immediate'],  # можно добавить linear
    'eval_batch': 256,
    'lr': 3e-4,
    'lr_schedule': 'constant',  # constant или warmup_linear
    'lr_start': 1e-4,
    'lr_end': 1e-5,
    'lr_warmup_epochs': 3,
    'weight_decay': 0.0,
    'seeds': [42],
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'save_epochs': [20, 21, 25, 50, 75],  # дополнительно к latest и final
    'save_every_epoch': False,  # True потребует больше диска
}
RESUME_DIR = None  # например PROJECT_ROOT / 'experiments/exp3/runs/2026...'
if CFG['smoke']:
    CFG.update(warmup_epochs=1, continue_epochs=2, batch_start=8,
               batch_end=16, eval_batch=7, save_epochs=[1])
device = torch.device(CFG['device'])
print('Устройство:', device, '| модель:', CFG['model'], '| профиль:', CFG['model_profile'])


### 4. Данные и фиксированное разделение

Данные масштабируются ToTensor в [0, 1], без случайных аугментаций. Состав validation одинаков для всех seeds. При batch-statistics BN фиксированы также порядок и batch оценки; остаток из одного примера для оценки запрещён.


In [ ]:
def build_datasets():
    if CFG['smoke']:
        g = torch.Generator().manual_seed(CFG['split_seed'])
        return tuple(TensorDataset(torch.rand(n, 3, 32, 32, generator=g),
            torch.randint(0, 10, (n,), generator=g)) for n in (33, 16))
    full = datasets.CIFAR10(CFG['data_root'], train=True,
        download=CFG['download'], transform=transforms.ToTensor())
    g = torch.Generator().manual_seed(CFG['split_seed'])
    indices = torch.randperm(len(full), generator=g).tolist()
    n_val = CFG['val_size']
    if not 0 < n_val < len(full):
        raise ValueError('Некорректный val_size.')
    return Subset(full, indices[n_val:]), Subset(full, indices[:n_val])

train_ds, val_ds = build_datasets()
CFG['batch_end'] = (round(CFG['lb_fraction'] * len(train_ds))
                    if CFG['batch_end'] is None else CFG['batch_end'])
assert CFG['warmup_epochs'] >= 0 and CFG['continue_epochs'] >= 1
assert 2 <= CFG['batch_start'] <= CFG['batch_end'] <= len(train_ds)
assert CFG['eval_batch'] >= 2
assert set(CFG['branches']) <= {'constant', 'linear', 'exponential', 'immediate'}
assert len(set(CFG['branches'])) == len(CFG['branches'])
assert len(set(CFG['seeds'])) == len(CFG['seeds'])
assert tuple(train_ds[0][0].shape) == (3, 32, 32)
for ds in (train_ds, val_ds):
    if len(ds) % CFG['eval_batch'] == 1 or len(ds) < 2:
        raise ValueError('Выбери eval_batch без остатка 1: это необходимо для batch-statistics BN.')
print('Train:', len(train_ds), '| validation:', len(val_ds))
print('Batch:', CFG['batch_start'], '→', CFG['batch_end'])


### 5. Модели: импорт проекта и совместимость

ShallowNN берётся из архива. notebook_c1 заменяет только pooling и BN и применяет прежнюю Xavier-инициализацию; project_native ничего не заменяет. Legacy VGG оставлена как явная опция для прежних экспериментов: DNN архива не эквивалентна ей.


In [ ]:
import math
import torch
from torch import nn


class PaperRepoVGG11BN(nn.Module):
    """VGG11-BN из PyTorch/vgg.py репозитория авторов.
    Вход: RGB-изображения 32 × 32.
    """
    def __init__(self, num_classes=10):
        super().__init__()

        architecture = [
            64, "M",
            128, "M",
            256, 256, "M",
            512, 512, "M",
            512, 512, "M",
        ]

        layers = []
        in_channels = 3

        for item in architecture:
            if item == "M":
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                layers.extend([
                    nn.Conv2d(in_channels, item, kernel_size=3, padding=1),
                    nn.BatchNorm2d(item),
                    nn.ReLU(inplace=True),
                ])
                in_channels = item

        self.features = nn.Sequential(*layers)

        # После пяти pooling: 32×32 → 1×1, получаем 512 признаков.
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(512, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

        self.initialize_weights()

    def initialize_weights(self):
        # Та же схема инициализации, что в репозитории.
        with torch.no_grad():
            for module in self.modules():
                if isinstance(module, nn.Conv2d):
                    n = (
                        module.kernel_size[0]
                        * module.kernel_size[1]
                        * module.out_channels
                    )
                    nn.init.normal_(module.weight, mean=0, std=math.sqrt(2 / n))
                    if module.bias is not None:
                        nn.init.zeros_(module.bias)

                elif isinstance(module, nn.BatchNorm2d):
                    nn.init.ones_(module.weight)
                    nn.init.zeros_(module.bias)

                elif isinstance(module, nn.Linear):
                    nn.init.normal_(module.weight, mean=0, std=0.01)
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)




class SameMaxPool2d(nn.Module):
    def forward(self, x):
        h, w = x.shape[-2:]
        ph = max(((h + 1) // 2 - 1) * 2 + 3 - h, 0)
        pw = max(((w + 1) // 2 - 1) * 2 + 3 - w, 0)
        x = nn.functional.pad(x, (pw//2, pw-pw//2, ph//2, ph-ph//2), value=float('-inf'))
        return nn.functional.max_pool2d(x, 3, 2)


def build_model(config=None):
    c = CFG if config is None else config
    classes = {'ShallowNN': ShallowNN, 'CNN': CNN, 'DNN': DNN}
    if c['model'] == 'legacy_vgg11_bn':
        return PaperRepoVGG11BN(c['num_classes'])
    if c['model'] not in classes:
        raise ValueError(c['model'])
    model = classes[c['model']](nb_classes=c['num_classes'])
    if c['model'] == 'ShallowNN':
        if c['model_profile'] == 'notebook_c1':
            model.pool1 = SameMaxPool2d()
            model.pool2 = SameMaxPool2d()
            for name, width in [('norm1', 64), ('norm2', 64), ('norm3', 384), ('norm4', 192)]:
                kind = nn.BatchNorm2d if name in ('norm1', 'norm2') else nn.BatchNorm1d
                setattr(model, name, kind(width, track_running_stats=False))
            for layer in model.modules():
                if isinstance(layer, (nn.Conv2d, nn.Linear)):
                    nn.init.xavier_uniform_(layer.weight)
                    nn.init.zeros_(layer.bias)
        elif c['model_profile'] != 'project_native':
            raise ValueError(c['model_profile'])
    return model


def build_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

probe = build_model().eval()
with torch.no_grad():
    assert probe(torch.zeros(2, 3, 32, 32)).shape == (2, CFG['num_classes'])
print('Параметров:', f'{sum(p.numel() for p in probe.parameters()):,}')
print('BN running stats:', [m.track_running_stats for m in probe.modules()
    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))])
del probe


### 6. Воспроизводимость, batch и LR

Одна версия каждого расписания. Веткам даётся одинаковый порядок примеров на соответствующей эпохе. Последний singleton объединяется с предыдущим batch; никакие примеры не отбрасываются. LR по умолчанию постоянный.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def cpu_copy(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu().clone()
    if isinstance(obj, dict):
        return {k: cpu_copy(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(cpu_copy(v) for v in obj)
    return copy.deepcopy(obj)


@contextmanager
def preserve_rng():
    py, np_state = random.getstate(), np.random.get_state()
    devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == 'cuda' else []
    try:
        with torch.random.fork_rng(devices=devices):
            yield
    finally:
        random.setstate(py)
        np.random.set_state(np_state)


def batch_for(branch, epoch):
    a, b = CFG['batch_start'], CFG['batch_end']
    fraction = (epoch - 1) / (CFG['continue_epochs'] - 1) if CFG['continue_epochs'] > 1 else 1.0
    if branch == 'constant': return a
    if branch == 'immediate': return b
    if branch == 'linear': return round(a + (b - a) * fraction)
    if branch == 'exponential': return round(a * (b / a) ** fraction)
    raise ValueError(branch)


def continuation_lr(epoch):
    if CFG['lr_schedule'] == 'constant': return CFG['lr']
    if CFG['lr_schedule'] != 'warmup_linear': raise ValueError(CFG['lr_schedule'])
    w, total = CFG['lr_warmup_epochs'], CFG['continue_epochs']
    if not (0 <= w < total) or w == 1: raise ValueError('warmup LR: 0 или >=2, меньше total.')
    if w and epoch <= w:
        return CFG['lr_start'] + (CFG['lr'] - CFG['lr_start']) * (epoch - 1) / (w - 1)
    frac = (epoch-w)/(total-w) if w else (epoch-1)/max(total-1, 1)
    return CFG['lr'] + (CFG['lr_end'] - CFG['lr']) * frac


def make_train_loader(batch_size, seed, epoch):
    g = torch.Generator().manual_seed(seed * 100000 + epoch)
    order = torch.randperm(len(train_ds), generator=g).tolist()
    batches = [order[i:i+batch_size] for i in range(0, len(order), batch_size)]
    if len(batches) > 1 and len(batches[-1]) == 1:
        tail = batches.pop()
        batches[-1].extend(tail)
    return DataLoader(train_ds, batch_sampler=batches, num_workers=0,
        generator=torch.Generator().manual_seed(seed + epoch))


### 7. Один шаг по эпохам и оценка из проекта

Обучение остаётся локальным: fit проекта имеет drop_last=True и не предоставляет нужного контроля веток/метрик. Оценка использует eval.evaluate и metrics.Accuracy из архива. Состояния RNG сохраняются вокруг оценки, чтобы она не влияла на Dropout последующего обучения.


In [ ]:
def train_epoch(model, optimizer, loader):
    model.train()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    count = steps = 0
    synchronize()
    start = time.perf_counter()
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = nn.functional.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        n = len(labels)
        loss_sum += loss.detach() * n
        correct += (logits.detach().argmax(1) == labels).sum()
        count += n
        steps += 1
    synchronize()
    seconds = time.perf_counter() - start
    return dict(train_loss_online=loss_sum.item()/count, train_acc_online=correct.item()/count,
                examples=count, steps=steps, train_seconds=seconds)


def measure(model, dataset):
    synchronize()
    start = time.perf_counter()
    with preserve_rng():
        values = project_evaluate(model, dataset, nn.CrossEntropyLoss(), str(device),
            batch_size=CFG['eval_batch'], metrics={'accuracy': Accuracy()})
    synchronize()
    return values, time.perf_counter() - start


def measure_both(model):
    val, vs = measure(model, val_ds)
    train, ts = measure(model, train_ds)
    return dict(val_loss=val['loss'], val_acc=val['accuracy'], val_seconds=vs,
        train_loss_eval=train['loss'], train_acc_eval=train['accuracy'], train_eval_seconds=ts)


### 8. Папка запуска и совместимость checkpoint

Новая папка создаётся ровно здесь. Для resume используется RESUME_DIR. Снимок конфигурации и хеши исходников защищают от случайного продолжения другой моделью. Повторное выполнение этой ячейки без RESUME_DIR создаёт новый эксперимент.


In [ ]:
def experiment_signature():
    c = {k: v for k, v in CFG.items() if k not in ('device', 'seeds', 'save_epochs', 'save_every_epoch')}
    sources = [PROJECT_ROOT/'models/instances'/f'{n}.py' for n in ('ShallowNN', 'CNN', 'DNN')]
    sources += [PROJECT_ROOT/'models/abs_model.py', PROJECT_ROOT/'eval.py', PROJECT_ROOT/'metrics/accuracy.py']
    hashes = {str(p.relative_to(PROJECT_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}
    payload = {'config': c, 'sources': hashes, 'protocol': 'exp3-clean-v1',
               'torch': torch.__version__, 'train_size': len(train_ds), 'val_size': len(val_ds)}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest(), payload

SIGNATURE, provenance = experiment_signature()
if RESUME_DIR is None:
    RUN_DIR = PROJECT_ROOT / 'experiments/exp3/runs' / (
        datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '_' + CFG['model'])
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    (RUN_DIR/'config.json').write_text(json.dumps(CFG, indent=2), encoding='utf-8')
    (RUN_DIR/'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
else:
    RUN_DIR = Path(RESUME_DIR).expanduser().resolve()
    if not (RUN_DIR/'provenance.json').is_file():
        raise ValueError('Автоматический resume только для папки этой чистой версии.')
    previous = json.loads((RUN_DIR/'provenance.json').read_text())
    expected = hashlib.sha256(json.dumps(previous, sort_keys=True).encode()).hexdigest()
    if expected != SIGNATURE:
        raise ValueError('Конфигурация/исходники изменились. Верни прежние настройки или создай новый запуск.')
print('Результаты:', RUN_DIR)


### 9. Checkpoint и CSV

После каждой завершённой эпохи атомарно сохраняется latest с весами, ADAM и историей. После сбоя повторный запуск продолжает со следующей эпохи. Итоговая модель и выбранные промежуточные эпохи хранятся отдельно; не требуется сохранять 100 полных копий.


In [ ]:
def atomic_save(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def atomic_csv(rows, path):
    path = Path(path)
    temporary = path.with_suffix('.csv.tmp')
    pd.DataFrame(rows).to_csv(temporary, index=False)
    temporary.replace(path)


def load_checkpoint(path):
    # Только собственные доверенные файлы эксперимента.
    ck = torch.load(path, map_location='cpu', weights_only=False)
    if ck.get('signature') != SIGNATURE:
        raise ValueError(f'Несовместимый checkpoint: {path}')
    return ck


def pack_checkpoint(model, optimizer, seed, branch, epoch, rows, **extra):
    return dict(model=cpu_copy(model.state_dict()), optimizer=cpu_copy(optimizer.state_dict()),
        config=copy.deepcopy(CFG), signature=SIGNATURE, seed=seed, branch=branch,
        epoch=epoch, rows=copy.deepcopy(rows), **extra)


def restore_model(ck):
    model = build_model().to(device)
    model.load_state_dict(ck['model'])
    optimizer = build_optimizer(model)
    optimizer.load_state_dict(copy.deepcopy(ck['optimizer']))
    return model, optimizer


def refresh_metrics():
    all_rows = []
    for seed in CFG['seeds']:
        rows = []
        for branch in CFG['branches']:
            path = RUN_DIR/f'{branch}_seed{seed}_latest.pt'
            if path.exists(): rows.extend(load_checkpoint(path)['rows'])
        if rows:
            atomic_csv(rows, RUN_DIR/f'metrics_seed{seed}.csv')
            all_rows.extend(rows)
    if all_rows: atomic_csv(all_rows, RUN_DIR/'metrics_all.csv')
    return pd.DataFrame(all_rows)


### 10. Предобучение и независимые продолжения

Каждая ветка восстанавливает один и тот же warmup и ADAM. Новые checkpoint совместимы только с этой версией notebook; старые файлы не подмешиваются. При прерывании незавершённая эпоха повторяется с тем же seed. Можно запускать только выбранные ветки.


In [ ]:
def ensure_warmup(seed):
    final = RUN_DIR/f'warmup_seed{seed}.pt'
    latest = RUN_DIR/f'warmup_seed{seed}_latest.pt'
    if final.exists(): return load_checkpoint(final)
    if latest.exists():
        ck = load_checkpoint(latest)
        model, optimizer = restore_model(ck)
        rows, completed = ck['rows'], ck['epoch']
    else:
        seed_everything(seed)
        model = build_model().to(device)
        optimizer = build_optimizer(model)
        rows, completed = [], 0
    for epoch in range(completed+1, CFG['warmup_epochs']+1):
        seed_everything(seed*100000+epoch)
        row = train_epoch(model, optimizer, make_train_loader(CFG['batch_start'], seed, epoch))
        row.update(measure_both(model))
        row.update(seed=seed, epoch=epoch, batch=CFG['batch_start'], lr=CFG['lr'])
        rows.append(row)
        ck = pack_checkpoint(model, optimizer, seed, 'warmup', epoch, rows)
        atomic_save(ck, latest)
        atomic_csv(rows, RUN_DIR/f'warmup_seed{seed}.csv')
        print(f"seed={seed} warmup={epoch} val_acc={row['val_acc']:.4f}", flush=True)
    baseline = measure_both(model)
    ck = pack_checkpoint(model, optimizer, seed, 'warmup', CFG['warmup_epochs'], rows,
        baseline=baseline, warmup_seconds=sum(r['train_seconds'] for r in rows))
    atomic_save(ck, final)
    return ck


def run_seed(seed, branches=None):
    if experiment_signature()[0] != SIGNATURE:
        raise ValueError('CFG изменён после создания папки; создай новый RUN_DIR через ячейку 8.')
    selected = CFG['branches'] if branches is None else list(branches)
    if not set(selected).issubset(CFG['branches']): raise ValueError(selected)
    warm = ensure_warmup(seed)
    for branch in selected:
        final = RUN_DIR/f'{branch}_seed{seed}.pt'
        latest = RUN_DIR/f'{branch}_seed{seed}_latest.pt'
        if final.exists():
            print(f'{branch}, seed={seed}: уже завершено, пропускаем.')
            continue
        if latest.exists():
            ck = load_checkpoint(latest)
            model, optimizer = restore_model(ck)
            rows, completed = ck['rows'], ck['epoch']
        else:
            model, optimizer = restore_model(warm)
            rows = [dict(seed=seed, branch=branch, epoch=0, **warm['baseline'],
                batch=CFG['batch_start'], lr=CFG['lr'], cumulative_train_seconds=0.,
                total_train_seconds=warm['warmup_seconds'], cumulative_steps=0)]
            completed = 0
        elapsed, steps = rows[-1]['cumulative_train_seconds'], rows[-1]['cumulative_steps']
        pair_path = RUN_DIR/f'{branch}_seed{seed}_worst_drop.pt'
        best_drop = torch.load(pair_path, map_location='cpu', weights_only=False)['drop'] if pair_path.exists() else 0.
        for epoch in range(completed+1, CFG['continue_epochs']+1):
            previous_state = cpu_copy(model.state_dict())
            absolute_epoch = CFG['warmup_epochs']+epoch
            seed_everything(seed*100000+absolute_epoch)
            batch = batch_for(branch, epoch)
            lr = continuation_lr(epoch)
            for group in optimizer.param_groups: group['lr'] = lr
            row = train_epoch(model, optimizer, make_train_loader(batch, seed, absolute_epoch))
            row.update(measure_both(model))
            elapsed += row['train_seconds']
            steps += row['steps']
            row.update(seed=seed, branch=branch, epoch=epoch, batch=batch, lr=lr,
                cumulative_train_seconds=elapsed, cumulative_steps=steps,
                total_train_seconds=warm['warmup_seconds']+elapsed)
            rows.append(row)
            ck = pack_checkpoint(model, optimizer, seed, branch, epoch, rows)
            atomic_save(ck, latest)
            if CFG['save_every_epoch'] or epoch in CFG['save_epochs']:
                atomic_save(ck, RUN_DIR/f'{branch}_seed{seed}_epoch{epoch}.pt')
            # Компактная пара весов крупнейшего падения для BN-диагностики.
            drop = rows[-2]['val_acc']-row['val_acc']
            if drop > best_drop:
                best_drop = drop
                atomic_save(dict(drop=drop, epoch=epoch, before_epoch=epoch-1, config=copy.deepcopy(CFG),
                    signature=SIGNATURE, model=cpu_copy(model.state_dict()), before_model=previous_state), pair_path)
            # Только маленькая CSV текущей ветки; общая таблица собирается ниже.
            atomic_csv(rows, RUN_DIR/f'{branch}_seed{seed}.csv')
            print(f"seed={seed} {branch:11s} epoch={epoch:3d} batch={batch:4d} lr={lr:.2e} "
                  f"train={row['train_acc_eval']:.4f} val={row['val_acc']:.4f}", flush=True)
        ck = pack_checkpoint(model, optimizer, seed, branch, CFG['continue_epochs'], rows)
        atomic_save(ck, final)
        del model, optimizer
        refresh_metrics()
    return refresh_metrics()


### 11. Проверка вместимости LB (необязательно)

Два полных шага отдельной модели; это техническая проверка, не часть результатов. После неё RNG восстанавливается. RUN_MEMORY_CHECK=True запускает проверку; по умолчанию выключено.


In [ ]:
RUN_MEMORY_CHECK = False
if RUN_MEMORY_CHECK:
    with preserve_rng():
        m = opt = x = y = loss = None
        try:
            m = build_model().to(device).train()
            opt = build_optimizer(m)
            if device.type == 'cuda': torch.cuda.reset_peak_memory_stats(device)
            x = torch.randn(CFG['batch_end'], 3, 32, 32, device=device)
            y = torch.randint(CFG['num_classes'], (len(x),), device=device)
            for _ in range(2):
                opt.zero_grad(set_to_none=True)
                loss = nn.functional.cross_entropy(m(x), y)
                loss.backward()
                opt.step()
            synchronize()
            print('LB помещается.')
            if device.type == 'cuda': print('Peak GiB:', torch.cuda.max_memory_allocated(device)/1024**3)
        finally:
            del m, opt, x, y, loss
            gc.collect()
            if device.type == 'cuda': torch.cuda.empty_cache()


### 12. Запуск / продолжение

Это единственная ячейка длительного обучения. Выполни её для старта. После Interrupt Kernel можно повторить: завершённые эпохи и ветки пропускаются. Для одной ветки поставь SELECTED_BRANCHES=["constant"]. После перезапуска kernel сначала выполни настройки с RESUME_DIR и определения выше.


In [ ]:
SELECTED_BRANCHES = None  # None — все; либо ['constant'], ['exponential', 'immediate']
try:
    for seed in CFG['seeds']:
        run_seed(seed, branches=SELECTED_BRANCHES)
finally:
    results = refresh_metrics()  # собирает сохранённые эпохи даже при прерывании
if not results.empty:
    display(results[results['epoch'] == CFG['continue_epochs']][[
        'seed', 'branch', 'val_acc', 'val_loss', 'train_acc_eval',
        'cumulative_steps', 'cumulative_train_seconds']])


### 13. Загрузить результаты без обучения

Используй для повторного построения графиков, в том числе старых запусков. Чтение CSV не меняет веса. Для другой папки измени RESULTS_DIR.


In [ ]:
RESULTS_DIR = RUN_DIR
results = pd.read_csv(Path(RESULTS_DIR)/'metrics_all.csv')
print(results.groupby(['seed', 'branch'])['epoch'].agg(['min', 'max', 'count']))


### 14. Графики по эпохам, шагам и времени

Сохраняются все прежние 14 панелей: train/validation, gaps, время, batch. Полоса — разброс между seeds, не между эпохами. Точки измерены только после эпох. Время исключает оценку и запись checkpoint; для сравнения полного времени нужны эти дополнительные расходы.


In [ ]:
colors = {
    "constant": "tab:blue",
    "exponential": "tab:orange",
    "linear": "tab:red",
    "immediate": "tab:green",
}

colors = {k: v for k, v in colors.items() if k in set(results["branch"])}
plot_data = results.copy()

required = {
    "train_loss_eval", "train_acc_eval",
    "val_loss", "val_acc", "cumulative_steps",
}
assert required.issubset(plot_data.columns), (
    f"В results отсутствуют: {required - set(plot_data.columns)}"
)

plot_data["accuracy_gap"] = (
    plot_data["train_acc_eval"] - plot_data["val_acc"]
)
plot_data["loss_gap"] = (
    plot_data["val_loss"] - plot_data["train_loss_eval"]
)

metrics = [
    ("train_acc_eval", "Train accuracy (eval)"),
    ("val_acc", "Validation accuracy"),
    ("train_loss_eval", "Train loss (eval)"),
    ("val_loss", "Validation loss"),
    ("accuracy_gap", "Accuracy gap: train − validation"),
    ("loss_gap", "Loss gap: validation − train"),
]

fig, axes = plt.subplots(7, 2, figsize=(15, 27))


def plot_metric(ax, metric, x_column, xlabel, ylabel):
    for branch, color in colors.items():
        part = plot_data[plot_data["branch"] == branch]

        # При одинаковом расписании batch точки по эпохам/шагам
        # совпадают между seeds: можно усреднять напрямую.
        stats = (
            part.groupby(x_column)[metric]
            .agg(["mean", "std"])
            .sort_index()
        )

        x = stats.index.to_numpy()
        mean = stats["mean"].to_numpy()
        std = stats["std"].fillna(0).to_numpy()

        ax.plot(
            x, mean,
            marker=".", color=color, label=branch,
        )

        if part["seed"].nunique() > 1:
            ax.fill_between(
                x, mean - std, mean + std,
                color=color, alpha=0.15,
            )

    ax.set(xlabel=xlabel, ylabel=ylabel)


for row, (metric, ylabel) in enumerate(metrics):
    plot_metric(
        axes[row, 0],
        metric,
        "epoch",
        "Continuation epoch",
        ylabel,
    )

    plot_metric(
        axes[row, 1],
        metric,
        "cumulative_steps",
        "Optimizer steps after checkpoint",
        ylabel,
    )

    if metric in {"accuracy_gap", "loss_gap"}:
        for ax in axes[row]:
            ax.axhline(
                0, color="black", linestyle=":", linewidth=1
            )

# Accuracy относительно времени обучения.
# Временные точки между seeds различаются:
# рисуем отдельные траектории без прямого усреднения.
for branch, color in colors.items():
    part = plot_data[plot_data["branch"] == branch]

    for index, (seed, run) in enumerate(part.groupby("seed")):
        run = run.sort_values("epoch")

        axes[6, 0].plot(
            run["cumulative_train_seconds"] / 60,
            run["val_acc"],
            color=color,
            alpha=0.7,
            label=branch if index == 0 else None,
        )

    schedule = (
        part[part["epoch"] > 0]
        .groupby("epoch")["batch"]
        .first()
        .sort_index()
    )

    axes[6, 1].plot(
        schedule.index,
        schedule.values,
        marker=".",
        color=color,
        label=branch,
    )

axes[6, 0].set(
    xlabel="Training time, minutes (excluding evaluation)",
    ylabel="Validation accuracy",
)
axes[6, 1].set(
    xlabel="Continuation epoch",
    ylabel="Batch size",
)

for ax in axes.flat:
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(
    Path(RESULTS_DIR) / "comparison_epochs_steps_time.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


### 15. Диагностика BatchNorm (необязательно)

Сохраняет прежнюю проверку BN без переобучения. Работает только для моделей с running statistics; для notebook_c1 с track_running_stats=False корректно пропускается. Можно выбрать final, epochN или worst_drop checkpoint. Старые C1 checkpoint имеют другие имена параметров и требуют явного переноса, автоматически не загружаются.


In [ ]:
RUN_BN_DIAGNOSTIC = False
BN_CHECKPOINT = RUN_DIR / f"constant_seed{CFG['seeds'][0]}.pt"

def bn_diagnostic(path, calibration_batch=256, state_key='model'):
    ck = load_checkpoint(path)
    with preserve_rng():
        m = build_model(ck['config']).to(device)
        m.load_state_dict(ck[state_key])
        layers = [x for x in m.modules() if isinstance(x, nn.modules.batchnorm._BatchNorm) and x.track_running_stats]
        if not layers:
            print('Накопленных статистик BN нет — пересчитывать нечего.')
            return pd.DataFrame()
        before = measure_both(m)
        original_params = {n: p.detach().cpu().clone() for n, p in m.named_parameters()}
        m.eval()
        momenta = [x.momentum for x in layers]
        for x in layers:
            x.reset_running_stats()
            x.momentum = None
            x.train()
        g = torch.Generator().manual_seed(12345)
        order = torch.randperm(len(train_ds), generator=g).tolist()
        batches = [order[i:i+calibration_batch] for i in range(0,len(order),calibration_batch)]
        if len(batches)>1 and len(batches[-1])==1:
            tail=batches.pop(); batches[-1].extend(tail)
        loader = DataLoader(train_ds, batch_sampler=batches,
            generator=torch.Generator().manual_seed(12345))
        try:
            with torch.no_grad():
                for x, _ in loader: m(x.to(device))
        finally:
            for x, momentum in zip(layers, momenta): x.momentum=momentum
            m.eval()
        assert all(torch.equal(p.detach().cpu(), original_params[n]) for n,p in m.named_parameters())
        after = measure_both(m)
        table = pd.DataFrame([before,after],index=['original','recalibrated'])
        table.to_csv(RUN_DIR/(Path(path).stem+'_'+state_key+'_bn.csv'))
        display(table[['train_acc_eval','train_loss_eval','val_acc','val_loss']])
        return table

if RUN_BN_DIAGNOSTIC:
    bn_result = bn_diagnostic(BN_CHECKPOINT)
    # Для worst_drop.pt: предыдущая эпоха хранится в before_model.
    ck_info = load_checkpoint(BN_CHECKPOINT)
    if 'before_model' in ck_info:
        print('Предыдущая эпоха:', ck_info['before_epoch'])
        bn_previous = bn_diagnostic(BN_CHECKPOINT, state_key='before_model')


### 16. Sharpness из архива (необязательно)

Подключена функция metrics.sharpness.compute_sharpness без изменения исходников. ВАЖНО: это не точная метрика 2.1 статьи: границы epsilon*tanh(z) не содержат масштаба |A⁺w|+1, используется LBFGS с преобразованием переменных, а не L-BFGS-B. Значения подписываются archive_sharpness. Нулевой старт и ограниченный поиск могут недооценивать максимум. Сначала небольшое фиксированное подмножество; выключено по умолчанию.


In [ ]:
RUN_SHARPNESS = False
SHARPNESS_SAMPLES = 1024
SHARPNESS_BATCH = 256
SHARPNESS_EPSILONS = [5e-4, 1e-3]
SHARPNESS_SUBSPACE = 20  # матрица A: n_parameters × subspace_dim, расход памяти!

if RUN_SHARPNESS:
    from metrics.sharpness import compute_sharpness
    count = min(SHARPNESS_SAMPLES, len(train_ds))
    if count < 2 or count % SHARPNESS_BATCH == 1:
        raise ValueError('Подмножество/batch не должны давать singleton.')
    g = torch.Generator().manual_seed(CFG['split_seed'])
    subset = Subset(train_ds, torch.randperm(len(train_ds),generator=g)[:count].tolist())
    sharp_rows=[]
    for seed in CFG['seeds']:
        for branch in ['warmup', *CFG['branches']]:
            path=RUN_DIR/f'{branch}_seed{seed}.pt'
            if not path.exists(): continue
            ck=load_checkpoint(path)
            m=build_model(ck['config']).to(device)
            m.load_state_dict(ck['model'])
            for epsilon in SHARPNESS_EPSILONS:
                with preserve_rng():
                    value=compute_sharpness(m,subset,str(device),batch_size=SHARPNESS_BATCH,
                        epsilon=epsilon,maxiter=10,subspace_dim=SHARPNESS_SUBSPACE,
                        seed=12345,workers=0,optimize=False)
                sharp_rows.append(dict(seed=seed,branch=branch,epsilon=epsilon,
                    samples=count,eval_batch=SHARPNESS_BATCH,subspace_dim=SHARPNESS_SUBSPACE,
                    archive_sharpness=value))
            del m
    sharpness_results=pd.DataFrame(sharp_rows)
    sharpness_results.to_csv(RUN_DIR/'archive_sharpness.csv',index=False)
    display(sharpness_results)


## Что сознательно отличается от общего API проекта

- Импортируются `ShallowNN`, `CNN`, `DNN`, `eval.evaluate`, `metrics.Accuracy`; sharpness — опционально.
- `train.fit` не заменяет цикл: drop_last=True, сброс параметров, фиксированный batch и отсутствие истории
  эпох изменили бы эксперимент. Наш цикл использует тот же стандартный PyTorch/ADAM, но сохраняет все примеры.
- `models.utils.save_model` сохраняет только state_dict. Для ветвления и resume нужны ADAM,
  номер эпохи, конфигурация и история; поэтому используется расширенный checkpoint.
- ShallowNN архива: симметричный pooling, стандартная инициализация и running-statistics BN.
  Профиль notebook_c1 адаптирует их к прежнему notebook без изменения файлов проекта.
- Старый VGG не заменён на DNN: их классификаторы различаются (4096→4096 против 512).
- Batch-statistics BN делает оценку зависимой от разбиения на batch. Между ветками оно фиксировано.
- Отдельные аварийные ячейки восстановления заменены единым resume; проверки BN сохранены отдельно.
- Старые outputs, pip-установка при каждом Run All, повторные CFG и переопределения функций удалены.
- Новый запуск не обязан побитово совпасть со старым: имена/порядок создания слоёв проекта отличаются.
  Старые метрики можно визуализировать, но старые checkpoint не используются автоматически.
